In [1]:
import pandas as pd, numpy as np
pd.set_option("display.max_columns", None)

raw = pd.read_csv("../data/raw/autodesk_materials.csv")

# Sy has text like "280 max" -> treat spec limits as missing, flag them
sy_text = raw["Sy"].astype(str).str.strip()
sy_is_max = sy_text.str.contains("max", case=False, na=False)
sy_num = pd.to_numeric(sy_text.where(~sy_is_max), errors="coerce")
leftover = sy_text[sy_num.isna() & ~sy_is_max]
print("Spec-limit Sy cells set to NaN:", sy_is_max.sum(), "| other unparseable:", len(leftover))

df = pd.DataFrame({
    "material":    raw["Material"].str.strip(),
    "standard":    raw["Std"],
    "condition":   raw["Heat treatment"],
    "description": raw["Desc"],
    "tensile_strength_MPa": raw["Su"].astype(float),
    "yield_strength_MPa":   sy_num,
    "elongation_pct":       raw["A5"],
    "youngs_modulus_GPa":   raw["E"] / 1000,      # MPa -> GPa
    "density_g_cm3":        raw["Ro"] / 1000,     # kg/m3 -> g/cm3
    "hardness_HV":          raw["HV"],
    "hardness_HB":          raw["Bhn"],
    "sy_was_spec_max":      sy_is_max,
})
df.describe().T.round(2)

Spec-limit Sy cells set to NaN: 8 | other unparseable: 0


,count,mean,std,min,25%,50%,75%,max
tensile_strength_MPa,1552.0,572.75,326.83,69.00,340.00,500.00,705.00,2220.00
yield_strength_MPa,1544.0,387.76,290.04,28.00,203.75,310.00,472.00,2048.00
elongation_pct,1346.0,19.33,12.42,0.50,11.00,16.00,24.38,70.00
youngs_modulus_GPa,1552.0,164.57,56.14,73.00,105.00,206.00,206.00,219.00
density_g_cm3,1552.0,6.93,2.12,1.75,7.16,7.86,7.86,8.93
hardness_HV,165.0,328.48,202.76,105.00,192.00,230.00,492.00,800.00
hardness_HB,463.0,177.14,113.51,19.00,78.00,170.00,229.00,627.00


In [16]:
import re

def tag_family(material, desc):
    t = f"{material} {'' if pd.isna(desc) else desc}".lower()
    if "alumin" in t and not any(k in t for k in ["bronze", "brass"]): return "Aluminium"
    if "magnes" in t:   return "Magnesium"
    if any(k in t for k in ["cast iron", "grey", "gray", "nodular", "malleable"]): return "Cast iron"
    if "stainless" in t: return "Stainless steel"
    if "steel" in t:     return "Steel"
    if any(k in t for k in ["bronze", "brass", "copper", "nickel silver", "muntz"]): return "Copper alloy"
    if re.search(r"\bc\d{5}\b", t): return "Copper alloy"   # UNS C-numbers are copper alloys
    return "Unlabelled"

df["family"] = [tag_family(m, d) for m, d in zip(df["material"], df["description"])]
df["is_stainless"] = (df["material"] + " " + df["description"].fillna("")).str.contains("stainless", case=False)

print(df["family"].value_counts())
print("\nSample of Unlabelled names:")
print("\nAll Unlabelled rows:")
print(df.loc[df["family"] == "Unlabelled", ["material", "standard", "description"]])

print("\nRows mentioning 'nickel':")
print(df.loc[df["material"].str.contains("nickel", case=False), ["material", "family"]])

family
Steel              818
Aluminium          251
Copper alloy       183
Stainless steel    150
Cast iron          113
Magnesium           37
Name: count, dtype: int64

Sample of Unlabelled names:

All Unlabelled rows:
Empty DataFrame
Columns: [material, standard, description]
Index: []

Rows mentioning 'nickel':
                     material        family
240      Copper-Nickel C96200  Copper alloy
241      Copper-Nickel C96300  Copper alloy
242      Copper-Nickel C96400  Copper alloy
243      Copper-Nickel C96600  Copper alloy
244      Nickel Silver C97300  Copper alloy
245      Nickel Silver C97400  Copper alloy
246      Nickel Silver C97600  Copper alloy
247      Nickel Silver C97800  Copper alloy
231  Nickel-Tin Bronze C94700  Copper alloy
232  Nickel-Tin Bronze C94800  Copper alloy


In [17]:
checks = {
    "yield > tensile":      df["yield_strength_MPa"] > df["tensile_strength_MPa"],
    "non-positive value":   (df[["tensile_strength_MPa","youngs_modulus_GPa","density_g_cm3"]] <= 0).any(axis=1),
    "elongation > 100%":    df["elongation_pct"] > 100,
}
for name, m in checks.items():
    print(f"{name}: {m.sum()} rows")
    if m.any():
        display(df.loc[m, ["material", "standard", "tensile_strength_MPa", "yield_strength_MPa", "elongation_pct"]].head(10))

yield > tensile: 2 rows


,material,standard,tensile_strength_MPa,yield_strength_MPa,elongation_pct
300,Copper Alloy C51100,ANSI,317.0,345.0,48.0
327,Copper Alloy C70400,ANSI,262.0,276.0,46.0


non-positive value: 0 rows
elongation > 100%: 0 rows


In [18]:
CORE = ["tensile_strength_MPa", "yield_strength_MPa", "elongation_pct",
        "youngs_modulus_GPa", "density_g_cm3"]

df["info_score"] = df.notna().sum(axis=1) + 2 * (df["family"] != "Unlabelled")
df = df.sort_values("info_score", ascending=False)
df["n_equivalents"] = df.groupby(CORE, dropna=False)["material"].transform("size")

df_u = df.drop_duplicates(subset=CORE, keep="first").sort_index().reset_index(drop=True)
print(f"Rows: {len(df)} -> {len(df_u)} after removing duplicates")
print(df_u["family"].value_counts())

Rows: 1552 -> 1261 after removing duplicates
family
Steel              687
Aluminium          197
Copper alloy       168
Stainless steel    117
Cast iron           62
Magnesium           30
Name: count, dtype: int64


In [19]:
has_h = df_u["hardness_HV"].notna() | df_u["hardness_HB"].notna()
print(df_u.groupby(has_h)[["tensile_strength_MPa", "elongation_pct"]].median().round(1))
print(pd.crosstab(df_u["family"], has_h, normalize="index").round(2))

       tensile_strength_MPa  elongation_pct
False                 491.0            19.0
True                  575.0            14.0
col_0            False  True 
family                       
Aluminium         0.34   0.66
Cast iron         0.29   0.71
Copper alloy      1.00   0.00
Magnesium         1.00   0.00
Stainless steel   0.73   0.27
Steel             0.54   0.46


In [20]:
print((df_u[CORE].isna().mean() * 100).round(1))
df_u.to_csv("../data/processed/materials_unique.csv", index=False)
print("Saved:", df_u.shape)

tensile_strength_MPa     0.0
yield_strength_MPa       0.2
elongation_pct          12.1
youngs_modulus_GPa       0.0
density_g_cm3            0.0
dtype: float64
Saved: (1261, 16)


In [21]:
CORE = ["tensile_strength_MPa", "yield_strength_MPa", "elongation_pct",
        "youngs_modulus_GPa", "density_g_cm3"]

print("=== Unlabelled rows ===")
print(df_u.loc[df_u["family"] == "Unlabelled", ["material", "standard", "description"]])

print("\n=== Nickel rows ===")
print(df_u.loc[df_u["material"].str.contains("nickel", case=False), ["material", "family"]])

print("\n=== Sanity checks (rows flagged, on the de-duplicated table) ===")
print("yield > tensile:", (df_u["yield_strength_MPa"] > df_u["tensile_strength_MPa"]).sum())
print("elongation > 100:", (df_u["elongation_pct"] > 100).sum())

print("\n=== Dedupe ===")
print("Rows before:", len(df), "-> after:", len(df_u))
print(df_u["family"].value_counts())
print("Stainless rows:", df_u["is_stainless"].sum())

print("\n=== Missing % on core features ===")
print((df_u[CORE].isna().mean() * 100).round(1))

print("\n=== Hardness missingness (Cell 4b) ===")
has_h = df_u["hardness_HV"].notna() | df_u["hardness_HB"].notna()
print("Rows with any hardness:", has_h.sum())
print(df_u.groupby(has_h)[["tensile_strength_MPa", "elongation_pct"]].median().round(1))
print(pd.crosstab(df_u["family"], has_h, normalize="index").round(2))

print("\n=== Summary of core features ===")
print(df_u[CORE].describe().T.round(2))

=== Unlabelled rows ===
Empty DataFrame
Columns: [material, standard, description]
Index: []

=== Nickel rows ===
                     material        family
219  Nickel-Tin Bronze C94700  Copper alloy
220  Nickel-Tin Bronze C94800  Copper alloy
228      Copper-Nickel C96200  Copper alloy
229      Copper-Nickel C96300  Copper alloy
230      Copper-Nickel C96400  Copper alloy
231      Copper-Nickel C96600  Copper alloy
232      Nickel Silver C97300  Copper alloy
233      Nickel Silver C97400  Copper alloy
234      Nickel Silver C97600  Copper alloy
235      Nickel Silver C97800  Copper alloy

=== Sanity checks (rows flagged, on the de-duplicated table) ===
yield > tensile: 2
elongation > 100: 0

=== Dedupe ===
Rows before: 1552 -> after: 1261
family
Steel              687
Aluminium          197
Copper alloy       168
Stainless steel    117
Cast iron           62
Magnesium           30
Name: count, dtype: int64
Stainless rows: 117

=== Missing % on core features ===
tensile_strength_MPa 

In [22]:
bad_yt = df_u["yield_strength_MPa"] > df_u["tensile_strength_MPa"]
print("Yield > tensile:")
display(df_u.loc[bad_yt, ["material", "standard", "condition", "tensile_strength_MPa", "yield_strength_MPa"]])
print("Missing yield:")
display(df_u.loc[df_u["yield_strength_MPa"].isna(), ["material", "standard", "tensile_strength_MPa"]])

n0 = len(df_u)
df_c = df_u[~bad_yt & df_u["yield_strength_MPa"].notna()].reset_index(drop=True)
print(f"\nDropped {n0 - len(df_c)} rows -> {len(df_c)} remain")

Yield > tensile:


,material,standard,condition,tensile_strength_MPa,yield_strength_MPa
288,Copper Alloy C51100,ANSI,NaN,317.0,345.0
314,Copper Alloy C70400,ANSI,NaN,262.0,276.0


Missing yield:


,material,standard,tensile_strength_MPa
589,DIN USt1303,DIN,270.0
590,DIN St1405,DIN,270.0
591,DIN St1203,DIN,270.0



Dropped 5 rows -> 1256 remain


In [23]:
miss_el = df_c["elongation_pct"].isna()
print("Fraction of elongation missing, by family:")
print(pd.crosstab(df_c["family"], miss_el, normalize="index").round(2))
print("\nMedian strength: rows with vs without elongation")
print(df_c.groupby(miss_el)[["tensile_strength_MPa", "yield_strength_MPa"]].median().round(0))

Fraction of elongation missing, by family:
elongation_pct   False  True 
family                       
Aluminium         0.91   0.09
Cast iron         0.35   0.65
Copper alloy      0.95   0.05
Magnesium         1.00   0.00
Stainless steel   1.00   0.00
Steel             0.87   0.13

Median strength: rows with vs without elongation
                tensile_strength_MPa  yield_strength_MPa
elongation_pct                                          
False                          517.0               310.0
True                           600.0               355.0


In [24]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: unlocks IterativeImputer
from sklearn.impute import IterativeImputer

CORE = ["tensile_strength_MPa", "yield_strength_MPa", "elongation_pct",
        "youngs_modulus_GPa", "density_g_cm3"]

# log10 + standardize so distances between rows are comparable across features
logX = np.log10(df_c[CORE])
scaler = StandardScaler().fit(logX)
Z = pd.DataFrame(scaler.transform(logX), columns=CORE, index=df_c.index)
fam = df_c["family"]

def to_df(arr, like):
    return pd.DataFrame(arr, columns=like.columns, index=like.index)

methods = {
    "mean":          lambda Zm, f: to_df(SimpleImputer(strategy="mean").fit_transform(Zm), Zm),
    "median":        lambda Zm, f: to_df(SimpleImputer(strategy="median").fit_transform(Zm), Zm),
    "family median": lambda Zm, f: Zm.fillna(Zm.groupby(f).transform("median")).fillna(Zm.median()),
    "KNN (k=5)":     lambda Zm, f: to_df(KNNImputer(n_neighbors=5).fit_transform(Zm), Zm),
    "KNN (k=10)":    lambda Zm, f: to_df(KNNImputer(n_neighbors=10).fit_transform(Zm), Zm),
    "Iterative":     lambda Zm, f: to_df(IterativeImputer(max_iter=20, random_state=0).fit_transform(Zm), Zm),
}

complete = Z.notna().all(axis=1)
Zc, famc = Z[complete], fam[complete]
print("Complete rows available:", len(Zc))

rng = np.random.default_rng(42)
n_hide = int(0.10 * len(Zc))
results = {m: [] for m in methods}

for trial in range(30):
    hide = rng.choice(Zc.index, size=n_hide, replace=False)   # rows whose elongation we hide
    Zm = Zc.copy()
    Zm.loc[hide, "elongation_pct"] = np.nan
    for name, fn in methods.items():
        imp = fn(Zm, famc)
        err = imp.loc[hide, "elongation_pct"] - Zc.loc[hide, "elongation_pct"]
        results[name].append(np.sqrt((err ** 2).mean()))

sd_log = np.log10(df_c.loc[complete, "elongation_pct"]).std(ddof=0)
summary = pd.DataFrame({
    "RMSE_mean": {m: np.mean(v) for m, v in results.items()},
    "RMSE_std":  {m: np.std(v)  for m, v in results.items()},
})
summary["typical error (x factor)"] = 10 ** (summary["RMSE_mean"] * sd_log)
summary.sort_values("RMSE_mean").round(3)

Complete rows available: 1104


,RMSE_mean,RMSE_std,typical error (x factor)
KNN (k=5),0.653,0.081,1.614
KNN (k=10),0.667,0.088,1.631
Iterative,0.747,0.099,1.730
family median,0.876,0.090,1.901
median,0.999,0.082,2.082
mean,1.000,0.081,2.082


In [25]:
BEST = "KNN (k=5)"      # <- change to the top row of YOUR summary table

Z_imp = methods[BEST](Z, fam)
filled = pd.DataFrame(10 ** scaler.inverse_transform(Z_imp), columns=CORE, index=Z.index)

df_c["elongation_imputed"] = df_c["elongation_pct"].isna()          # flag BEFORE filling
df_c["elongation_pct"] = df_c["elongation_pct"].fillna(filled["elongation_pct"])

print("Imputed cells:", int(df_c["elongation_imputed"].sum()))
print("Missing left in core features:", int(df_c[CORE].isna().sum().sum()))
print(df_c["family"].value_counts())
display(df_c[CORE].describe().T.round(2))

df_c.to_csv("../data/processed/materials_clean.csv", index=False)
print("Saved:", df_c.shape)

Imputed cells: 152
Missing left in core features: 0
family
Steel              684
Aluminium          197
Copper alloy       166
Stainless steel    117
Cast iron           62
Magnesium           30
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
tensile_strength_MPa,1256.0,587.72,341.17,69.00,350.00,520.00,731.25,2220.00
yield_strength_MPa,1256.0,397.00,306.39,28.00,200.00,315.00,487.00,2048.00
elongation_pct,1256.0,18.62,12.30,0.50,10.00,15.71,23.00,70.00
youngs_modulus_GPa,1256.0,165.62,55.81,73.00,105.00,206.00,206.00,219.00
density_g_cm3,1256.0,6.98,2.11,1.75,7.79,7.86,7.86,8.93


Saved: (1256, 17)


In [26]:
ci = df_c[df_c["family"] == "Cast iron"].copy()
ci["subtype"] = np.select(
    [ci["material"].str.contains("grey|gray", case=False, na=False),
     ci["material"].str.contains("nodular", case=False, na=False),
     ci["material"].str.contains("malleable", case=False, na=False)],
    ["Grey", "Nodular", "Malleable"], default="other")

print(pd.crosstab(ci["subtype"], ci["elongation_imputed"]))
print()
print(ci.groupby(["subtype", "elongation_imputed"])["elongation_pct"]
        .describe()[["count", "min", "50%", "max"]].round(1))

elongation_imputed  False  True 
subtype                         
Grey                    0     21
Malleable              13      4
Nodular                 8      6
other                   1      9

                              count  min   50%   max
subtype   elongation_imputed                        
Grey      True                 21.0  1.7  21.7  34.6
Malleable False                13.0  1.5   4.0  12.0
          True                  4.0  2.6   5.4  12.7
Nodular   False                 8.0  2.0   5.0  22.0
          True                  6.0  2.0   4.1   7.3
other     False                 1.0  4.0   4.0   4.0
          True                  9.0  2.0   7.3  23.1


In [27]:
grey = df_c["material"].str.contains("grey|gray", case=False, na=False)
mask = grey & df_c["elongation_imputed"]

df_c["elongation_assumed"] = False
df_c.loc[mask, "elongation_pct"] = 0.5       # brittle: ~no plastic elongation (see note)
df_c.loc[mask, "elongation_assumed"] = True

print("Grey iron rows set by assumption:", mask.sum())
print("KNN-imputed rows remaining:", int((df_c["elongation_imputed"] & ~df_c["elongation_assumed"]).sum()))
df_c.to_csv("../data/processed/materials_clean.csv", index=False)

Grey iron rows set by assumption: 21
KNN-imputed rows remaining: 131


In [28]:
ci = df_c[df_c["family"] == "Cast iron"].copy()
ci["subtype"] = np.select(
    [ci["material"].str.contains("grey|gray", case=False, na=False),
     ci["material"].str.contains("nodular", case=False, na=False),
     ci["material"].str.contains("malleable", case=False, na=False)],
    ["Grey", "Nodular", "Malleable"], default="other")

ci["status"] = np.select(
    [ci["elongation_assumed"], ci["elongation_imputed"]],
    ["assumed 0.5", "KNN-imputed"], default="measured")

print(pd.crosstab(ci["subtype"], ci["status"]))
print()
print(ci.groupby(["subtype", "status"])["elongation_pct"]
        .describe()[["count", "min", "50%", "max"]].round(1))

status     KNN-imputed  assumed 0.5  measured
subtype                                      
Grey                 0           21         0
Malleable            4            0        13
Nodular              6            0         8
other                9            0         1

                       count  min  50%   max
subtype   status                            
Grey      assumed 0.5   21.0  0.5  0.5   0.5
Malleable KNN-imputed    4.0  2.6  5.4  12.7
          measured      13.0  1.5  4.0  12.0
Nodular   KNN-imputed    6.0  2.0  4.1   7.3
          measured       8.0  2.0  5.0  22.0
other     KNN-imputed    9.0  2.0  7.3  23.1
          measured       1.0  4.0  4.0   4.0


In [15]:
print(tag_family("Steel 304", "Stainless steel"))     # expect: Stainless steel
print(tag_family("Special Alloy C99300", None))       # expect: Copper alloy

Stainless steel
Copper alloy
